This project builds a chatbot using a transformer model to generate responses. It takes user input, processes it, and provides meaningful conversational output.

In [8]:
import os
# Disable ALL progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()


In [11]:
!pip install transformers torch --quiet

In [12]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Device map: Single device


In [13]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [14]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    # Initialize chat history with a system message to set behavior
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue
        # Generate reply from the model
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        # fallback
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hi
Chatbot: Hello! How can I help you today?
User: what is NLP?
Chatbot: NLP stands for Natural Language Processing. It's a field of artificial intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable computers to understand, interpret, and generate human language in a way that is useful for specific applications. Some common uses of NLP include speech recognition, machine translation, sentiment analysis, and text summarization.
User: what is AI?
Chatbot: AI, or Artificial Intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. The ultimate goal is to create machines that work and react like humans. AI can be appli